# Phase 3b: Calibration fix (per-label Platt scaling)

Problem found in Phase 3: at high confidence the model is overconfident ("97%" is right ~59% of the time), because it was trained on balanced data (48% toxic) but real data is ~10% toxic.
Fix: for each of the 6 labels, fit 2 numbers (scale a, shift b) on **validation** data: `p = sigmoid(a * score + b)`. Ranking is unchanged, so every AUC stays identical.

**Settings:** GPU T4 ×2 · Internet On · Secret HF_TOKEN · Inputs:
1. **Toxic Comment Classification Challenge**
2. **Phase 2c notebook output** (full-test scores)
3. **Phase 3 notebook output** (to recalibrate the Gemini race log; optional)

**Save Version → Save & Run All** (~25 min).

In [1]:
# ==== CONFIG ====
FT_REPO = "Vinay57/laya-jigsaw-moderation"
BATCH = 64
PUSH = True    # upload calibration_platt.json to the HF repo

In [2]:
!pip install -q -U laya
import os, glob, json, time, numpy as np, pandas as pd, torch, laya
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
OUT = "/kaggle/working/demo_data_v2"; PLOTS = "/kaggle/working/plots_v2"
os.makedirs(OUT, exist_ok=True); os.makedirs(PLOTS, exist_ok=True)
print("laya", laya.__version__, "| CUDA", torch.cuda.is_available())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.7/79.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.5/214.5 kB 5.6 MB/s eta 0:00:00
laya 0.3.21 | CUDA True


## 1. Data: all validation comments (for fitting), test (for checking)

In [3]:
from sklearn.model_selection import train_test_split
LABELS = ['toxic','severe_toxic','obscene','threat','insult','identity_hate']
QUESTIONS = {
  'toxic':         {'type':'noul','instructions':'Is this comment rude, disrespectful or unreasonable?'},
  'severe_toxic':  {'type':'noul','instructions':'Is this comment extremely hateful, aggressive or disrespectful?'},
  'obscene':       {'type':'noul','instructions':'Does this comment contain obscene or vulgar language?'},
  'threat':        {'type':'noul','instructions':'Does this comment threaten to harm someone?'},
  'insult':        {'type':'noul','instructions':'Does this comment insult a person?'},
  'identity_hate': {'type':'noul','instructions':'Does this comment attack someone for their identity (race, religion, gender, sexuality, etc.)?'},
}
MAX_CHARS = 1200
BASE = os.path.dirname(glob.glob('/kaggle/input/**/train.csv*', recursive=True)[0])
def load(name):
    for ext in ['.csv', '.csv.zip']:
        p = f'{BASE}/{name}{ext}'
        if os.path.exists(p): return pd.read_csv(p)
    raise FileNotFoundError(name)
train = load('train')
test = load('test').merge(load('test_labels'), on='id')
test = test[test['toxic'] != -1].reset_index(drop=True)
train = train.drop_duplicates('comment_text').reset_index(drop=True)
tr, va = train_test_split(train, test_size=0.1, random_state=42, stratify=train['toxic'])  # same split as always
sample = test.sample(10000, random_state=42)
print("validation", len(va), va[LABELS].sum().to_dict())

validation 15958 {'toxic': 1529, 'severe_toxic': 149, 'obscene': 847, 'threat': 50, 'insult': 800, 'identity_hate': 153}


## 2. Load merged model + direct scoring

In [4]:
from huggingface_hub import snapshot_download, HfApi
from kaggle_secrets import UserSecretsClient
from safetensors.torch import load_file
from transformers import AutoTokenizer
from laya.agent import _fix_tokenizer_config
from laya.common import build_model, build_sequence, QTYPES
from sklearn.metrics import roc_auc_score

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
NOUL = QTYPES["noul"]
d = snapshot_download(FT_REPO, token=HF_TOKEN); _fix_tokenizer_config(d)
cfg = json.load(open(os.path.join(d, "rl_agent_config.json")))
T = float(cfg["temperature"][NOUL])
tok = AutoTokenizer.from_pretrained(os.path.join(d, "tokenizer"))
model = build_model(cfg, encoder_dir=os.path.join(d, "encoder"))
model.load_state_dict(load_file(os.path.join(d, "model.safetensors")), strict=True)
model = model.to("cuda").eval()
print("current temperature:", T)

def build_all(texts, labels=LABELS):
    seqs = []
    for i, text in enumerate(texts):
        state = {'comment': str(text)[:MAX_CHARS]}
        for l in labels:
            ids, mk = build_sequence(tok, state, {"t": "noul", "ins": QUESTIONS[l]["instructions"], "crit": {}},
                                     cfg["max_len"], cfg["head_max_len"])
            seqs.append((i, LABELS.index(l), ids, mk))
    return seqs

def collate(items):
    n, L = len(items), max(len(s) for s, _ in items); kmax = max(len(m) for _, m in items)
    ids = torch.full((n, L), tok.pad_token_id, dtype=torch.long); att = torch.zeros((n, L), dtype=torch.long)
    mpos = torch.zeros((n, kmax), dtype=torch.long); mmask = torch.zeros((n, kmax), dtype=torch.bool)
    for i, (s, m) in enumerate(items):
        ids[i, :len(s)] = torch.tensor(s); att[i, :len(s)] = 1
        mpos[i, :len(m)] = torch.tensor(m); mmask[i, :len(m)] = True
    return ids.cuda(), att.cuda(), mpos.cuda(), mmask.cuda(), torch.full((n,), NOUL, dtype=torch.long).cuda()

@torch.no_grad()
def score(seqs, n_rows):
    order = sorted(range(len(seqs)), key=lambda k: len(seqs[k][2]))
    margin = np.full((n_rows, len(LABELS)), -1e9); finish = np.zeros(n_rows)
    torch.cuda.synchronize(); t0 = time.time()
    for b in range(0, len(order), BATCH):
        idx = order[b:b + BATCH]
        with torch.autocast("cuda", dtype=torch.float16):
            logits, _ = model(*collate([(seqs[k][2], seqs[k][3]) for k in idx]))
        lg = logits.float()[:, :2].cpu().numpy(); now = time.time() - t0
        for r, k in enumerate(idx):
            i, j = seqs[k][0], seqs[k][1]
            margin[i, j] = lg[r, 1] - lg[r, 0]; finish[i] = max(finish[i], now)
    torch.cuda.synchronize()
    return margin, time.time() - t0, finish

sig = lambda x: 1 / (1 + np.exp(-x))

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

current temperature: 1.1033602860690965


## 3. Score validation and fit the per-label correction

In [5]:
vm, secs, _ = score(build_all(va.comment_text.tolist()), len(va))
yv = va[LABELS].values
print(f"validation scored: {len(va)} comments in {secs/60:.1f} min")

from sklearn.linear_model import LogisticRegression
PLATT = {}
for j, l in enumerate(LABELS):
    lr = LogisticRegression(C=1e4, max_iter=2000).fit(vm[:, [j]], yv[:, j])
    PLATT[l] = {"a": float(lr.coef_[0, 0]), "b": float(lr.intercept_[0])}
    assert PLATT[l]["a"] > 0, "scale must be positive to keep ranking"
print(pd.DataFrame(PLATT).T.round(3).to_string())

def calibrate(m):   # raw scores -> calibrated probabilities
    return np.column_stack([sig(PLATT[l]["a"] * m[:, j] + PLATT[l]["b"]) for j, l in enumerate(LABELS)])
def old_prob(m):
    return sig(m / T)

W0928 15:47:40.976000 25 torch/_inductor/utils.py:1679] [1/0_1] Not enough SMs to use max_autotune_gemm mode


validation scored: 15958 comments in 13.4 min
                   a      b
toxic          0.875 -2.006
severe_toxic   0.749 -2.759
obscene        0.851 -1.927
threat         0.775 -3.252
insult         0.848 -1.388
identity_hate  0.731 -2.280


## 4. Check on test: before vs after

In [6]:
fp = glob.glob('/kaggle/input/**/merged_test_margins.csv', recursive=True)
if fp:
    tm = pd.read_csv(fp[0]).values; ty = test[LABELS].values; set_name = f"full test ({len(test):,})"
else:
    tm = None; print("Phase 2c output not attached - will use the 10k sample after it is scored")

sm, s_secs, s_finish = score(build_all(sample.comment_text.tolist()), len(sample))   # also the new stream recording
ys = sample[LABELS].values
if tm is None:
    tm, ty, set_name = sm, ys, "10k test sample"

def ece_pos(p, y, bins=10):
    p, y = p.ravel(), y.ravel(); edges = np.linspace(0, 1, bins + 1); e = 0.0
    for i, (lo, hi) in enumerate(zip(edges[:-1], edges[1:])):
        m = (p >= lo) & ((p < hi) if i < bins - 1 else (p <= hi))
        if m.any(): e += m.mean() * abs(p[m].mean() - y[m].mean())
    return e
def bins_of(p, y, bins=10):
    p, y = p.ravel(), y.ravel(); edges = np.linspace(0, 1, bins + 1); rows = []
    for i, (lo, hi) in enumerate(zip(edges[:-1], edges[1:])):
        m = (p >= lo) & ((p < hi) if i < bins - 1 else (p <= hi))
        if m.any(): rows.append({"bin": f"{lo:.1f}-{hi:.1f}", "mean_pred": float(p[m].mean()), "observed": float(y[m].mean()), "n": int(m.sum())})
    return rows

P0, P1 = old_prob(tm), calibrate(tm)
rows = []
for j, l in enumerate(LABELS):
    hi0, hi1 = P0[:, j] >= 0.9, P1[:, j] >= 0.9
    rows.append({"label": l,
                 "ECE before": ece_pos(P0[:, j], ty[:, j]), "ECE after": ece_pos(P1[:, j], ty[:, j]),
                 "Brier before": np.mean((P0[:, j] - ty[:, j]) ** 2), "Brier after": np.mean((P1[:, j] - ty[:, j]) ** 2),
                 "right when >=90% (before)": ty[hi0, j].mean() if hi0.any() else np.nan,
                 "right when >=90% (after)": ty[hi1, j].mean() if hi1.any() else np.nan,
                 "AUC before": roc_auc_score(ty[:, j], P0[:, j]), "AUC after": roc_auc_score(ty[:, j], P1[:, j])})
cal_tab = pd.DataFrame(rows).set_index("label")
cal_tab.loc["all (pooled)", ["ECE before", "ECE after"]] = [ece_pos(P0, ty), ece_pos(P1, ty)]
print(f"On {set_name}:"); print(cal_tab.round(4).to_string())

fig, ax = plt.subplots(1, 2, figsize=(11, 5))
for a_ in ax: a_.plot([0, 1], [0, 1], "k--", lw=1, label="perfect")
b0, b1 = bins_of(P0, ty), bins_of(P1, ty)
ax[0].plot([r["mean_pred"] for r in b0], [r["observed"] for r in b0], "o-", label=f"before (temperature only), ECE {ece_pos(P0, ty):.3f}")
ax[0].plot([r["mean_pred"] for r in b1], [r["observed"] for r in b1], "o-", label=f"after (per-label Platt), ECE {ece_pos(P1, ty):.3f}")
ax[0].set_title(f"All 6 labels pooled - {set_name}")
for j, l in enumerate(LABELS):
    bl = bins_of(P1[:, j], ty[:, j]); ax[1].plot([r["mean_pred"] for r in bl], [r["observed"] for r in bl], "o-", ms=4, label=l)
ax[1].set_title("After fix, per label")
for a_ in ax:
    a_.set_xlabel("Predicted probability"); a_.set_ylabel("Actual fraction positive"); a_.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{PLOTS}/calibration_fixed.png", dpi=150); plt.show()

json.dump({"method": "per-label Platt scaling fitted on validation", "platt": PLATT, "set": set_name,
           "table": cal_tab.round(5).reset_index().to_dict("records"), "pooled_bins_before": b0, "pooled_bins_after": b1},
          open(f"{OUT}/calibration.json", "w"), indent=1)
json.dump({"questions": QUESTIONS, "platt": PLATT, "max_chars": MAX_CHARS}, open(f"{OUT}/platt.json", "w"), indent=1)

On full test (63,978):
               ECE before  ECE after  Brier before  Brier after  right when >=90% (before)  right when >=90% (after)  AUC before  AUC after
label                                                                                                                                      
toxic              0.1241     0.0531        0.1024       0.0535                     0.5565                    0.8826      0.9726     0.9726
severe_toxic       0.0266     0.0037        0.0167       0.0050                     0.2920                       NaN      0.9887     0.9887
obscene            0.0560     0.0177        0.0456       0.0257                     0.6588                    0.9571      0.9828     0.9828
threat             0.0128     0.0010        0.0063       0.0021                     0.3872                       NaN      0.9962     0.9962
insult             0.0351     0.0072        0.0305       0.0226                     0.7866                    0.9517      0.9813     0.98

## 5. Routing with calibrated probabilities

In [7]:
pt, yt = P1[:, 0], ty[:, 0]
rows = []
for lo in [0.02, 0.05, 0.1, 0.2]:
    for hi in [0.8, 0.9, 0.95]:
        rem, app = pt >= hi, pt <= lo; hum = ~(rem | app)
        rows.append({"approve_below": lo, "remove_above": hi,
                     "auto_removed_%": 100 * rem.mean(), "auto_approved_%": 100 * app.mean(), "to_human_%": 100 * hum.mean(),
                     "removal_precision_%": 100 * yt[rem].mean() if rem.any() else np.nan,
                     "toxic_missed_in_approved_%": 100 * yt[app].mean() if app.any() else np.nan})
route = pd.DataFrame(rows)
print(route.round(2).to_string(index=False))
route.to_json(f"{OUT}/routing_table.json", orient="records", indent=1)
json.dump({"p": np.round(pt, 4).tolist(), "y": yt.astype(int).tolist()}, open(f"{OUT}/routing_points.json", "w"))

plt.figure(figsize=(6, 4))
for lo in [0.02, 0.05, 0.1]:
    s = route[route.approve_below == lo]; plt.plot(s["to_human_%"], s["removal_precision_%"], "o-", label=f"approve below {lo}")
plt.xlabel("% sent to human review"); plt.ylabel("Precision of auto-removals (%)"); plt.title("Routing trade-off (calibrated)")
plt.legend(); plt.tight_layout(); plt.savefig(f"{PLOTS}/routing_fixed.png", dpi=150); plt.show()

 approve_below  remove_above  auto_removed_%  auto_approved_%  to_human_%  removal_precision_%  toxic_missed_in_approved_%
          0.02          0.80            9.43            73.68       16.88                72.29                        0.17
          0.02          0.90            4.07            73.68       22.24                88.26                        0.17
          0.02          0.95            0.00            73.68       26.31               100.00                        0.17
          0.05          0.80            9.43            76.16       14.41                72.29                        0.26
          0.05          0.90            4.07            76.16       19.76                88.26                        0.26
          0.05          0.95            0.00            76.16       23.84               100.00                        0.26
          0.10          0.80            9.43            77.79       12.78                72.29                        0.34
          0.10  

## 6. New stream recording + recalibrated race log

In [8]:
SP = calibrate(sm)
stream = [{"text": str(t)[:280].replace("\n", " "), "y": y.astype(int).tolist(), "p": np.round(p, 3).tolist(), "t": round(float(f), 3)}
          for t, y, p, f in zip(sample.comment_text, ys, SP, s_finish)]
stream.sort(key=lambda x: x["t"])
json.dump({"device": "Tesla T4 (Kaggle, free)", "questions_per_comment": 6, "calibration": "per-label Platt",
           "comments_per_sec": len(sample) / s_secs, "comments": stream}, open(f"{OUT}/stream_log.json", "w"))
print(f"stream: {len(stream)} comments, {len(sample)/s_secs:.1f} comments/sec")

rl = glob.glob('/kaggle/input/**/race_log.json', recursive=True)
if rl:
    race = json.load(open(rl[0]))
    rm, _, _ = score(build_all([x["text"] for x in race]), len(race))
    rp = calibrate(rm)
    for x, p in zip(race, rp): x["laya"]["p"] = np.round(p, 3).tolist()   # latency (ms) unchanged
    json.dump(race, open(f"{OUT}/race_log.json", "w"))
    print("race log recalibrated:", len(race))
else:
    print("Phase 3 output not attached - race log not recalibrated")

stream: 10000 comments, 20.0 comments/sec
race log recalibrated: 100


## 7. HateCheck again with calibrated probabilities (flag if >= 0.5)

In [9]:
try:
    from datasets import load_dataset
    ds = load_dataset("Paul/hatecheck"); hc = ds[list(ds.keys())[0]].to_pandas()
    hy = (hc["label_gold"] == "hateful").astype(int).values
    hm, _, _ = score(build_all(hc["test_case"].tolist(), labels=["identity_hate"]), len(hc))
    j = LABELS.index("identity_hate")
    hp = sig(PLATT["identity_hate"]["a"] * hm[:, j] + PLATT["identity_hate"]["b"]); fl = hp >= 0.5
    hsum = {"AUC": roc_auc_score(hy, hm[:, j]), "accuracy": float((fl == hy).mean()),
            "hateful caught": float(fl[hy == 1].mean()), "non-hateful wrongly flagged": float(fl[hy == 0].mean())}
    print({k: round(v, 3) for k, v in hsum.items()})
    json.dump(hsum, open(f"{OUT}/hatecheck_calibrated.json", "w"), indent=1)
except Exception as e:
    print("HateCheck skipped:", e)

README.md: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/3728 [00:00<?, ? examples/s]

{'AUC': np.float64(0.805), 'accuracy': 0.6, 'hateful caught': 0.461, 'non-hateful wrongly flagged': 0.094}


## 8. Upload the calibration file to the HF repo

In [10]:
if PUSH:
    HfApi(token=HF_TOKEN).upload_file(path_or_fileobj=f"{OUT}/platt.json", path_in_repo="calibration_platt.json",
                                      repo_id=FT_REPO, commit_message="Add per-label Platt calibration (fitted on validation)")
    print("uploaded calibration_platt.json")
for f in sorted(os.listdir(OUT)): print(f, f"{os.path.getsize(os.path.join(OUT, f))/1e6:.2f} MB")

uploaded calibration_platt.json
calibration.json 0.00 MB
hatecheck_calibrated.json 0.00 MB
platt.json 0.00 MB
race_log.json 0.04 MB
routing_points.json 0.70 MB
routing_table.json 0.00 MB
stream_log.json 2.87 MB
